In [1]:
# =====================================================
# PROYECTO: Sistema de Recomendación de Películas
# METODOLOGÍA: CRISP-DM - Fase X
# REPRODUCIBILIDAD: Semillas fijas + chequeo hardware
# =====================================================

import random
import numpy as np
import polars as pl
import psutil
import matplotlib.pyplot as plt
from matplotlib import pyplot 
import seaborn as sns
from wordcloud import WordCloud
import os
from pathlib import Path
import urllib.request
import zipfile
from pathlib import Path

# Importing necessary modules for model evaluation and preprocessing 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB

# Importing GridSearchCV for hyperparameter tuning and mean_squared_error for model evaluation
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

# package that will help us to save tough models, this will allow us to run all notebook quickly 
import pickle as pk
import pickle
from math import sqrt
import json
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# ===================== REPRODUCIBILIDAD =====================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pl.Config.set_tbl_rows(10)          # para no imprimir todo
pl.Config.set_tbl_cols(20)

# ===================== CHEQUEO HARDWARE =====================
def check_hardware():
    ram_gb = psutil.virtual_memory().available / (1024**3)
    cpu_cores = psutil.cpu_count(logical=True)
    print(f"RAM disponible: {ram_gb:.1f} GB")
    print(f"CPU núcleos: {cpu_cores}")
    
    if ram_gb < 12:
        print("⚠️ ADVERTENCIA: RAM insuficiente (<12 GB). Puede fallar con 25M datos.")
    elif ram_gb < 20:
        print("⚠️ Recomendado: 20+ GB RAM para procesar completo sin problemas.")
    else:
        print("✅ Hardware suficiente para MovieLens 25M.")

check_hardware()

print(f"Python version: {os.sys.version}")
print(f"Polars version: {pl.__version__}")

RAM disponible: 1.7 GB
CPU núcleos: 8
⚠️ ADVERTENCIA: RAM insuficiente (<12 GB). Puede fallar con 25M datos.
Python version: 3.12.2 (v3.12.2:6abddd9f6a, Feb  6 2024, 17:02:06) [Clang 13.0.0 (clang-1300.0.29.30)]
Polars version: 1.9.0


In [2]:
# Descarga automática del dataset (reproducible)
data_path = Path("../data")
dataset_dir = data_path / "ml-25m"
ratings_file = dataset_dir / "ratings.csv"
movies_file = dataset_dir / "movies.csv"
tags_file = dataset_dir / "tags.csv"
links_file = dataset_dir / "links.csv"
genome_scores_file = dataset_dir / "genome-scores.csv"
genome_tags_file = dataset_dir / "genome-tags.csv"

# 2. Comprobar si los CSV (el dataset) existen
# Si falta alguno, procedemos a descargar y extraer
if not ratings_file.exists() or not movies_file.exists() or not tags_file.exists() or not links_file.exists() or not genome_scores_file.exists() or not genome_tags_file.exists():
    print("Dataset no encontrado. Descargando ml-25m.zip...")
    data_path.mkdir(parents=True, exist_ok=True)
    zip_path = data_path / "ml-25m.zip"
    
    # Descarga
    url = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
    urllib.request.urlretrieve(url, zip_path)
    
    print("Descarga completada. Extrayendo archivos...")
    # Extracción
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_path)
        
    print("Extracción completada con éxito.")
    
    zip_path.unlink()
else:
    print("El dataset ya existe en la carpeta. Omitiendo descarga.")

print("-" * 40)

# 3. Carga con Polars
# Pasamos las rutas que ya definimos arriba
ratings = pl.scan_csv(ratings_file)
movies = pl.scan_csv(movies_file)
tags = pl.scan_csv(tags_file)
links = pl.scan_csv(links_file)
genome_scores = pl.scan_csv(genome_scores_file)
genome_tags = pl.scan_csv(genome_tags_file)


El dataset ya existe en la carpeta. Omitiendo descarga.
----------------------------------------


#Análisis de datos en ratings 

In [3]:
print("Tipos de datos en ratings:")
print(ratings.schema)
print(f"Filas estimadas ratings: {ratings.select(pl.len()).collect().item():,}")

# 4. Estadísticas rápidas
stats = ratings.select([
    pl.col("rating").mean().alias("avg_rating"),
    pl.col("rating").min().alias("min_rating"),
    pl.col("rating").max().alias("max_rating"),
    pl.col("userId").n_unique().alias("usuarios_unicos")
]).collect()

print(stats)
print(ratings.describe())


Tipos de datos en ratings:
Schema({'userId': Int64, 'movieId': Int64, 'rating': Float64, 'timestamp': Int64})


/var/folders/zc/251h96fj3672hyd2fk9sprq80000gn/T/ipykernel_26797/3392182850.py:2: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(ratings.schema)


Filas estimadas ratings: 25,000,095
shape: (1, 4)
┌────────────┬────────────┬────────────┬─────────────────┐
│ avg_rating ┆ min_rating ┆ max_rating ┆ usuarios_unicos │
│ ---        ┆ ---        ┆ ---        ┆ ---             │
│ f64        ┆ f64        ┆ f64        ┆ u32             │
╞════════════╪════════════╪════════════╪═════════════════╡
│ 3.533854   ┆ 0.5        ┆ 5.0        ┆ 162541          │
└────────────┴────────────┴────────────┴─────────────────┘
shape: (9, 5)
┌────────────┬──────────────┬──────────────┬─────────────┬──────────────┐
│ statistic  ┆ userId       ┆ movieId      ┆ rating      ┆ timestamp    │
│ ---        ┆ ---          ┆ ---          ┆ ---         ┆ ---          │
│ str        ┆ f64          ┆ f64          ┆ f64         ┆ f64          │
╞════════════╪══════════════╪══════════════╪═════════════╪══════════════╡
│ count      ┆ 2.5000095e7  ┆ 2.5000095e7  ┆ 2.5000095e7 ┆ 2.5000095e7  │
│ null_count ┆ 0.0          ┆ 0.0          ┆ 0.0         ┆ 0.0          │
│ mea

#Análisis de datos en movies
print("Tipos de datos en movies:")
print(movies.schema)
print(f"Filas estimadas en movies: {movies.select(pl.len()).collect().item():,}")
print(movies.describe())


#Análisis de datos en tags

#Análisis de datos en genome-scores